In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

In [ ]:
!pip install kaleido==0.2.1 -q

In [ ]:
df = pd.read_csv("df_final.csv", parse_dates=["timestamp"])

In [ ]:
df.head()

,timestamp,house_1,house_2,house_3,house_4,house_5,house_6,house_7,house_8,hour,minute,time_str,weekday,is_weekend,month,season
0,2017-11-11 00:30:00,58.17,22.40,16.62,38.96,26.560,60.30,84.49,32.90,0,30,00:30,5,True,11,осень
1,2017-11-11 01:00:00,49.74,19.84,14.28,33.84,22.585,55.80,74.94,29.15,1,0,01:00,5,True,11,осень
2,2017-11-11 01:30:00,44.76,17.84,12.78,32.94,20.960,49.74,66.70,24.95,1,30,01:30,5,True,11,осень
3,2017-11-11 02:00:00,38.22,17.32,12.06,30.30,18.800,49.56,60.67,24.60,2,0,02:00,5,True,11,осень
4,2017-11-11 02:30:00,35.40,15.28,11.40,27.98,18.165,47.64,57.23,23.20,2,30,02:30,5,True,11,осень


In [ ]:
df["hour"] = df["timestamp"].dt.hour
df["minute"] = df["timestamp"].dt.minute
df["time_str"] = df["timestamp"].dt.strftime("%H:%M")
df["weekday"] = df["timestamp"].dt.weekday
df["is_weekend"] = df["weekday"] >= 5
df["month"] = df["timestamp"].dt.month
df["season"] = df["month"].map({
    12: "зима", 1: "зима", 2: "зима",
    3: "весна", 4: "весна", 5: "весна",
    6: "лето", 7: "лето", 8: "лето",
    9: "осень", 10: "осень", 11: "осень"
})

In [ ]:
houses = ["house_1", "house_2", "house_3", "house_4",
          "house_5", "house_6", "house_7", "house_8"]

In [ ]:
colors = {
    "house_1": "#1f77b4", "house_2": "#ff7f0e",
    "house_3": "#2ca02c", "house_4": "#d62728",
    "house_5": "#9467bd", "house_6": "#8c564b",
    "house_7": "#e377c2", "house_8": "#7f7f7f"
}

In [ ]:
df_norm = df.copy()
for house in houses:
    house_max = df[house].max()
    df_norm[house] = df[house] / house_max

In [ ]:
fig = make_subplots(
    rows=4, cols=2,
    subplot_titles=houses,
    vertical_spacing=0.10,
    horizontal_spacing=0.08
)

for i, house in enumerate(houses):
    row = i // 2 + 1
    col = i % 2 + 1

    workday = df_norm[df_norm["is_weekend"] == False].groupby("time_str")[house].mean()
    weekend = df_norm[df_norm["is_weekend"] == True].groupby("time_str")[house].mean()

    fig.add_trace(go.Scatter(
        x=workday.index, y=workday.values,
        mode="lines", name="рабочий",
        line=dict(color="#1f77b4", width=1.5, shape="hv"),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=weekend.index, y=weekend.values,
        mode="lines", name="Выходной",
        line=dict(color="#d62728", width=1.5, shape="hv"),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.update_xaxes(title_text="Время", row=row, col=col, title_font=dict(size=9))
    fig.update_yaxes(title_text="Отн. нагрузка", row=row, col=col, title_font=dict(size=9))

fig.update_layout(
    title="Суточный график нагрузки рабочих и выходных дней",
    width=700, height=1100,
    font=dict(size=10),
    margin=dict(l=50, r=20, t=60, b=40),
    hovermode="x unified",
    legend=dict(font=dict(size=9))
)

fig.show()
fig.write_image("07_daily_profile_workday_weekend.png", scale=2)

In [ ]:
df_norm["mean_all"] = df_norm[houses].mean(axis=1)

workday = df_norm[df_norm["is_weekend"] == False].groupby("time_str")["mean_all"].mean()
weekend = df_norm[df_norm["is_weekend"] == True].groupby("time_str")["mean_all"].mean()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=workday.index, y=workday.values,
    mode="lines", name="рабочий",
    line=dict(color="#1f77b4", width=2, shape="hv")
))

fig.add_trace(go.Scatter(
    x=weekend.index, y=weekend.values,
    mode="lines", name="выходной",
    line=dict(color="#d62728", width=2, shape="hv")
))

fig.update_layout(
    title="Усредненный суточный график нагрузки по всем домам",
    xaxis_title="Время",
    yaxis_title="Относительная нагрузка",
    width=700, height=350,
    font=dict(size=11),
    margin=dict(l=50, r=20, t=40, b=40),
    hovermode="x unified",
    legend=dict(font=dict(size=9))
)

fig.show()
fig.write_image("08_daily_profile_avg.png", scale=2)

In [ ]:
season_colors = {
    "зима": "#4e79a7",
    "весна": "#59a14f",
    "лето": "#f28e2b",
    "осень": "#e15759"
}

df_norm["mean_all"] = df_norm[houses].mean(axis=1)

#рабочие дни
fig_work = go.Figure()

for season in ["зима", "весна", "лето", "осень"]:
    profile = df_norm[
        (df_norm["season"] == season) &
        (df_norm["is_weekend"] == False)
    ].groupby("time_str")["mean_all"].mean()

    fig_work.add_trace(go.Scatter(
        x=profile.index, y=profile.values,
        mode="lines", name=season,
        line=dict(color=season_colors[season], width=2, shape="hv")
    ))

fig_work.update_layout(
    title="Суточный график нагрузок по сезонам для рабочих дней",
    xaxis_title="Время", yaxis_title="Относительная нагрузка",
    width=700, height=350, font=dict(size=11),
    margin=dict(l=50, r=20, t=40, b=40),
    hovermode="x unified", legend=dict(font=dict(size=9))
)
fig_work.show()
fig_work.write_image("09_daily_profile_seasons_workday.png", scale=2)

#выходные дни
fig_week = go.Figure()

for season in ["зима", "весна", "лето", "осень"]:
    profile = df_norm[
        (df_norm["season"] == season) &
        (df_norm["is_weekend"] == True)
    ].groupby("time_str")["mean_all"].mean()

    fig_week.add_trace(go.Scatter(
        x=profile.index, y=profile.values,
        mode="lines", name=season,
        line=dict(color=season_colors[season], width=2, shape="hv")
    ))

fig_week.update_layout(
    title="Суточный график нагрузки по сезонам для выходных дней",
    xaxis_title="Время", yaxis_title="Относительная нагрузка",
    width=700, height=350, font=dict(size=11),
    margin=dict(l=50, r=20, t=40, b=40),
    hovermode="x unified", legend=dict(font=dict(size=9))
)
fig_week.show()
fig_week.write_image("09_daily_profile_seasons_weekend.png", scale=2)

In [ ]:
df_norm["mean_all"] = df_norm[houses].mean(axis=1)

day_names = {
    0: "Пн", 1: "Вт", 2: "Ср", 3: "Чт", 4: "Пт", 5: "Сб", 6: "Вс"
}

weekly_p95 = df_norm.groupby("weekday")["mean_all"].quantile(0.95)
weekly_p05 = df_norm.groupby("weekday")["mean_all"].quantile(0.05)

weekly_p95.index = [day_names[i] for i in weekly_p95.index]
weekly_p05.index = [day_names[i] for i in weekly_p05.index]

#95-й перцентиль
fig_max = go.Figure()
fig_max.add_trace(go.Scatter(
    x=weekly_p95.index, y=weekly_p95.values,
    mode="lines+markers",
    line=dict(color="#d62728", width=2),
    marker=dict(size=7),
    showlegend=False
))
fig_max.update_layout(
    title="95-й перцентиль нагрузки по дням недели",
    xaxis_title="День недели", yaxis_title="Относительная нагрузка",
    width=700, height=350, font=dict(size=11),
    margin=dict(l=50, r=20, t=40, b=40)
)
fig_max.show()
fig_max.write_image("10_weekly_p95.png", scale=2)

#5-й перцентиль
fig_min = go.Figure()
fig_min.add_trace(go.Scatter(
    x=weekly_p05.index, y=weekly_p05.values,
    mode="lines+markers",
    line=dict(color="#1f77b4", width=2),
    marker=dict(size=7),
    showlegend=False
))
fig_min.update_layout(
    title="5-й перцентиль нагрузки по дням недели",
    xaxis_title="День недели", yaxis_title="Относительная нагрузка",
    width=700, height=350, font=dict(size=11),
    margin=dict(l=50, r=20, t=40, b=40)
)
fig_min.show()
fig_min.write_image("10_weekly_p05.png", scale=2)

95-й перцентиль — воскресенье явный пик, пятница минимум среди рабочих дней
5-й перцентиль — воскресенье минимум, понедельник максимум
Логично — в воскресенье люди дома весь день, поэтому и пики выше и ночные минимумы ниже.

In [ ]:
df_norm["date"] = df_norm["timestamp"].dt.date

df_daily_max = df_norm.groupby("date")[houses].max()
df_daily_min = df_norm.groupby("date")[houses].min()

df_daily_max["weekday"] = pd.to_datetime(df_daily_max.index).weekday
df_daily_min["weekday"] = pd.to_datetime(df_daily_min.index).weekday

weekly_max = df_daily_max.groupby("weekday")[houses].mean()
weekly_min = df_daily_min.groupby("weekday")[houses].mean()

weekly_max.index = [day_names[i] for i in weekly_max.index]
weekly_min.index = [day_names[i] for i in weekly_min.index]

#максимум
fig_max = go.Figure()
for house in houses:
    fig_max.add_trace(go.Scatter(
        x=weekly_max.index, y=weekly_max[house].values,
        mode="lines+markers",
        name=house,
        line=dict(color=colors[house], width=1.5),
        marker=dict(size=5)
    ))
fig_max.update_layout(
    title="Средний дневной максимум по дням недели (нормировано)",
    xaxis_title="День недели", yaxis_title="Относительная нагрузка",
    width=700, height=350, font=dict(size=11),
    margin=dict(l=50, r=20, t=40, b=40),
    legend=dict(font=dict(size=9))
)
fig_max.show()
fig_max.write_image("13_weekly_daily_max.png", scale=2)

#минимум нагрузки
fig_min = go.Figure()
for house in houses:
    fig_min.add_trace(go.Scatter(
        x=weekly_min.index, y=weekly_min[house].values,
        mode="lines+markers",
        name=house,
        line=dict(color=colors[house], width=1.5),
        marker=dict(size=5)
    ))
fig_min.update_layout(
    title="Средний дневной минимум по дням недели (нормировано)",
    xaxis_title="День недели", yaxis_title="Относительная нагрузка",
    width=700, height=350, font=dict(size=11),
    margin=dict(l=50, r=20, t=40, b=40),
    legend=dict(font=dict(size=9))
)
fig_min.show()
fig_min.write_image("13_weekly_daily_min.png", scale=2)

In [ ]:
df_norm["hour"] = df_norm["timestamp"].dt.hour

heatmap_data = df_norm.groupby(["weekday", "hour"])["mean_all"].mean().unstack()
heatmap_data.index = [day_names[i] for i in heatmap_data.index]

fig = go.Figure(go.Heatmap(
    z=heatmap_data.values,
    x=heatmap_data.columns,
    y=heatmap_data.index,
    colorscale="RdYlBu_r",
    colorbar=dict(title="Отн. нагрузка", thickness=15)
))

fig.update_layout(
    title="Тепловая карта нагрузки: час × день недели",
    xaxis_title="Час",
    yaxis_title="День недели",
    width=700, height=350,
    font=dict(size=11),
    margin=dict(l=50, r=20, t=40, b=40)
)
fig.show()
fig.write_image("11_heatmap_hour_weekday.png", scale=2)

ночь 0-6 — синий, минимум потребления
вечер 18-22 — красный, пик потребления
воскресенье — заметно теплее днём чем рабочие дни

In [ ]:
month_names = {
    1: "Янв", 2: "Фев", 3: "Мар", 4: "Апр",
    5: "Май", 6: "Июн", 7: "Июл", 8: "Авг",
    9: "Сен", 10: "Окт", 11: "Ноя", 12: "Дек"
}

heatmap_data2 = df_norm.groupby(["month", "hour"])["mean_all"].mean().unstack()
heatmap_data2.index = [month_names[i] for i in heatmap_data2.index]

fig2 = go.Figure(go.Heatmap(
    z=heatmap_data2.values,
    x=heatmap_data2.columns,
    y=heatmap_data2.index,
    colorscale="RdYlBu_r",
    colorbar=dict(title="Отн. нагрузка", thickness=15)
))

fig2.update_layout(
    title="Тепловая карта нагрузки: час × месяц",
    xaxis_title="Час",
    yaxis_title="Месяц",
    width=700, height=400,
    font=dict(size=11),
    margin=dict(l=50, r=20, t=40, b=40)
)
fig2.show()
fig2.write_image("12_heatmap_hour_month.png", scale=2)

зима (дек-фев) — тёмно-красный вечером, высокое потребление
лето (июн-авг) — значительно холоднее, низкое потребление
ночь 0-6 — синий везде, независимо от месяца
вечерний пик 19-22 — ярко выражен зимой